In [1]:

import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("Device:", device)
print("Current folder:", os.getcwd())


PyTorch version: 2.14.1
Device: cpu
Current folder: /Users/hussainbohra/Downloads


In [2]:
#Step 2 — Load the dataset


# Set the dataset path
data_dir = os.path.expanduser("~/Downloads/images_dataSAT")

# Check that both class folders exist
print("Dataset folder exists:", os.path.isdir(data_dir))

if os.path.isdir(data_dir):
    print("Dataset folders:", os.listdir(data_dir))
else:
    print("Dataset not found. Check the folder path.")


Dataset folder exists: True
Dataset folders: ['class_1_agri', 'class_0_non_agri', 'Module1_Data_Loading.ipynb']


In [3]:
#Step 3 — Load and split the images
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

dataset = datasets.ImageFolder(
    root=data_dir,
    transform=transform
)

# Split: 80% training, 20% validation
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True
)

val_loader = DataLoader(
    val_dataset, batch_size=32, shuffle=False
)

print("Classes:", dataset.classes)
print("Total images:", len(dataset))
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))

Classes: ['class_0_non_agri', 'class_1_agri']
Total images: 6000
Training images: 4800
Validation images: 1200


In [4]:
#Step 4: Create the PyTorch CNN model

class CNN(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


model = CNN().to(device)

print(model)

CNN(
  (features): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (9): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (10): ReLU()
    (11): AdaptiveAvgPool2d(output_size=(1, 1))
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=128, out_features=64, bias=True)
    (2): ReLU()
    (3): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [5]:
#Step 5: Set up the loss function and optimizer
criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss function:", criterion)
print("Optimizer:", optimizer)

Loss function: BCEWithLogitsLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [6]:
# Step 6 — Train the model

num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    print(
        f"Epoch [{epoch + 1}/{num_epochs}], "
        f"Loss: {epoch_loss:.4f}"
    )


Epoch [1/5], Loss: 0.2145
Epoch [2/5], Loss: 0.0715
Epoch [3/5], Loss: 0.0418
Epoch [4/5], Loss: 0.0746
Epoch [5/5], Loss: 0.0327


In [7]:
#Step 7: Evaluate the trained model
model.eval()

all_labels = []
all_preds = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)

        outputs = model(images)

        # BCEWithLogitsLoss outputs raw logits
        preds = (outputs >= 0).int().view(-1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print("Total validation images:", len(all_labels))
print("Total predictions:", len(all_preds))
print("Actual classes:", np.unique(all_labels))
print("Predicted classes:", np.unique(all_preds))

Total validation images: 1200
Total predictions: 1200
Actual classes: [0 1]
Predicted classes: [0 1]


In [8]:
#Step 8: Calculate the evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, zero_division=0)
recall = recall_score(all_labels, all_preds, zero_division=0)
f1 = f1_score(all_labels, all_preds, zero_division=0)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

print("\nClassification Report:")
print(classification_report(
    all_labels,
    all_preds,
    target_names=dataset.classes,
    zero_division=0
))

Accuracy:  0.9917
Precision: 0.9863
Recall:    0.9965
F1-score:  0.9914

Confusion Matrix:
[[616   8]
 [  2 574]]

Classification Report:
                  precision    recall  f1-score   support

class_0_non_agri       1.00      0.99      0.99       624
    class_1_agri       0.99      1.00      0.99       576

        accuracy                           0.99      1200
       macro avg       0.99      0.99      0.99      1200
    weighted avg       0.99      0.99      0.99      1200



In [9]:
# Step 9: Save the PyTorch results

pytorch_results = {
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-score": f1
}

print("PyTorch Model Results")
for metric, value in pytorch_results.items():
    print(f"{metric}: {value:.4f}")

PyTorch Model Results
Accuracy: 0.9917
Precision: 0.9863
Recall: 0.9965
F1-score: 0.9914


In [10]:

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

model.eval()
all_labels = []
all_preds = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        # BCEWithLogitsLoss returns logits, so 0 is the correct threshold.
        preds = (outputs >= 0).int().view(-1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

print("Accuracy:", accuracy_score(all_labels, all_preds))

print("\nConfusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

print("\nClassification Report:")
print(classification_report(
    all_labels,
    all_preds,
    target_names=["class_0_non_agri", "class_1_agri"],
    zero_division=0
))


Accuracy: 0.9916666666666667

Confusion Matrix:
[[616   8]
 [  2 574]]

Classification Report:
                  precision    recall  f1-score   support

class_0_non_agri       1.00      0.99      0.99       624
    class_1_agri       0.99      1.00      0.99       576

        accuracy                           0.99      1200
       macro avg       0.99      0.99      0.99      1200
    weighted avg       0.99      0.99      0.99      1200



In [11]:
import numpy as np

pytorch_results = {
    "Accuracy": accuracy_score(all_labels, all_preds),
    "Precision": precision_score(
        all_labels, all_preds, zero_division=0
    ),
    "Recall": recall_score(
        all_labels, all_preds, zero_division=0
    ),
    "F1-score": f1_score(
        all_labels, all_preds, zero_division=0
    )
}

print("Final PyTorch Results")
for metric, value in pytorch_results.items():
    print(f"{metric}: {value:.4f}")

np.savez(
    "pytorch_validation_results.npz",
    y_true=np.array(all_labels),
    y_pred=np.array(all_preds)
)

print("PyTorch results saved successfully.")

Final PyTorch Results
Accuracy: 0.9917
Precision: 0.9863
Recall: 0.9965
F1-score: 0.9914
PyTorch results saved successfully.


In [12]:
# Task 1: Explain what preds > 0.5 does
print("Task 1:")
print("It converts predicted probabilities into binary class labels.")
print("Values greater than or equal to 0.5 become class 1; values below 0.5 become class 0.")

# Task 3: Explain the significance of F1-score
print("\nTask 3:")
print("F1-score is the harmonic mean of precision and recall.")
print("It balances both metrics, especially when classes are imbalanced.")

# Task 5: Count false negatives from the PyTorch confusion matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])
true_negatives, false_positives, false_negatives, true_positives = cm.ravel()

print("\nTask 5:")
print("PyTorch confusion matrix:")
print(cm)
print("Total false negatives:", false_negatives)

Task 1:
It converts predicted probabilities into binary class labels.
Values greater than or equal to 0.5 become class 1; values below 0.5 become class 0.

Task 3:
F1-score is the harmonic mean of precision and recall.
It balances both metrics, especially when classes are imbalanced.

Task 5:
PyTorch confusion matrix:
[[616   8]
 [  2 574]]
Total false negatives: 2


In [13]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

def print_metrics(y_true, y_pred, model_name):
    print(f"\\n{model_name} Performance Metrics")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"F1-score:  {f1_score(y_true, y_pred, zero_division=0):.4f}")

# PyTorch metrics from the current validation predictions
print_metrics(all_labels, all_preds, "PyTorch")

# Keras confusion matrix from the earlier Keras evaluation:
# [[TN, FP], [FN, TP]] = [[568, 10], [7, 615]]
# Reconstruct label arrays from these counts so print_metrics reproduces the recorded Keras metrics.
keras_cm = np.array([[568, 10], [7, 615]])
keras_y_true = np.concatenate([
    np.zeros(keras_cm[0].sum(), dtype=int),
    np.ones(keras_cm[1].sum(), dtype=int)
])
keras_y_pred = np.concatenate([
    np.concatenate([np.zeros(568, dtype=int), np.ones(10, dtype=int)]),
    np.concatenate([np.zeros(7, dtype=int), np.ones(615, dtype=int)])
])
print_metrics(keras_y_true, keras_y_pred, "Keras")


\nPyTorch Performance Metrics
Accuracy:  0.9917
Precision: 0.9863
Recall:    0.9965
F1-score:  0.9914
\nKeras Performance Metrics
Accuracy:  0.9858
Precision: 0.9840
Recall:    0.9887
F1-score:  0.9864


In [14]:

from sklearn.metrics import confusion_matrix

# PyTorch confusion matrix
pytorch_cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])

print("PyTorch Confusion Matrix:")
print(pytorch_cm)

# False negatives: actual class 1 predicted as class 0
false_negatives = pytorch_cm[1, 0]
print("PyTorch False Negatives:", false_negatives)


PyTorch Confusion Matrix:
[[616   8]
 [  2 574]]
PyTorch False Negatives: 2
